In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from zipfile import ZipFile

zip_path = "/content/drive/MyDrive/output.zip"
extract_path = "/content"

with ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Zip açıldı ✅")

Zip açıldı ✅


In [ ]:
# 3) GEREKLİ AYARLAR + GENERATOR
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator

DATA_DIR = "/content/output"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

classes = sorted(os.listdir(TRAIN_DIR))
num_classes = len(classes)

print("Sınıflar:", classes)
print("Sınıf sayısı:", num_classes)
print("GPU:", tf.config.list_physical_devices("GPU"))

train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    classes=classes,
    class_mode="categorical",
    color_mode="rgb",
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    classes=classes,
    class_mode="categorical",
    color_mode="rgb",
    shuffle=False
)

Sınıflar: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'No_leaf', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus', 'Tomato___healthy']
Sınıf sayısı: 35
GPU: [Physica

In [ ]:
# 4) 10 EPOCH BİTMİŞ MODELİ YÜKLE
# Model Drive'da neredeyse yolu ona göre değiştir
MODEL_PATH = "/content/output/models/efficientnet_no_leaf_best.keras"

model = tf.keras.models.load_model(MODEL_PATH)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

checkpoint = keras.callbacks.ModelCheckpoint(
    "/content/drive/MyDrive/efficientnet_no_leaf_best_15epoch.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

In [ ]:
# 5) 11. EPOCH'TAN 15'E KADAR DEVAM ET
history = model.fit(
    train_generator,
    epochs=15,
    initial_epoch=10,
    validation_data=val_generator,
    callbacks=[checkpoint, early_stop]
)

print("Devam eğitimi bitti ✅")

Epoch 11/15
 632/1770 ━━━━━━━━━━━━━━━━━━━━ 8:18 438ms/step - accuracy: 0.9380 - loss: 0.1983

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


1770/1770 ━━━━━━━━━━━━━━━━━━━━ 0s 437ms/step - accuracy: 0.9371 - loss: 0.2001
Epoch 11: val_loss improved from None to 0.10998, saving model to /content/drive/MyDrive/efficientnet_no_leaf_best_15epoch.keras

Epoch 11: finished saving model to /content/drive/MyDrive/efficientnet_no_leaf_best_15epoch.keras
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 859s 468ms/step - accuracy: 0.9362 - loss: 0.2005 - val_accuracy: 0.9682 - val_loss: 0.1100
Epoch 12/15
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 0s 414ms/step - accuracy: 0.9357 - loss: 0.2007
Epoch 12: val_loss improved from 0.10998 to 0.10886, saving model to /content/drive/MyDrive/efficientnet_no_leaf_best_15epoch.keras

Epoch 12: finished saving model to /content/drive/MyDrive/efficientnet_no_leaf_best_15epoch.keras
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 773s 437ms/step - accuracy: 0.9372 - loss: 0.1977 - val_accuracy: 0.9688 - val_loss: 0.1089
Epoch 13/15
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 0s 420ms/step - accuracy: 0.9395 - loss: 0.1894
Epoch 13: val_loss improved from 0

In [ ]:
model = tf.keras.models.load_model(
    "/content/output/models/efficientnet_no_leaf_best_15epoch.keras"
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

checkpoint = keras.callbacks.ModelCheckpoint(
    "/content/output/models/efficientnet_no_leaf_best_20epoch.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history3 = model.fit(
    train_generator,
    epochs=20,
    initial_epoch=15,
    validation_data=val_generator,
    callbacks=[checkpoint, early_stop]
)

Epoch 16/20
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 0s 419ms/step - accuracy: 0.9425 - loss: 0.1814
Epoch 16: val_loss improved from None to 0.10128, saving model to /content/output/models/efficientnet_no_leaf_best_20epoch.keras

Epoch 16: finished saving model to /content/output/models/efficientnet_no_leaf_best_20epoch.keras
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 816s 449ms/step - accuracy: 0.9412 - loss: 0.1818 - val_accuracy: 0.9711 - val_loss: 0.1013
Epoch 17/20
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 0s 420ms/step - accuracy: 0.9421 - loss: 0.1801
Epoch 17: val_loss improved from 0.10128 to 0.10127, saving model to /content/output/models/efficientnet_no_leaf_best_20epoch.keras

Epoch 17: finished saving model to /content/output/models/efficientnet_no_leaf_best_20epoch.keras
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 782s 442ms/step - accuracy: 0.9420 - loss: 0.1809 - val_accuracy: 0.9706 - val_loss: 0.1013
Epoch 18/20
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 0s 413ms/step - accuracy: 0.9410 - loss: 0.1816
Epoch 18: val_loss imp

In [ ]:
!cp "/content/output/models/efficientnet_no_leaf_best_20epoch.keras" "/content/drive/MyDrive/"

In [ ]:
import tensorflow as tf

MODEL_PATH = "/content/output/models/efficientnet_no_leaf_best_20epoch.keras"

model = tf.keras.models.load_model(MODEL_PATH)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open("/content/model.tflite", "wb") as f:
    f.write(tflite_model)

print("model.tflite oluşturuldu ✅")

Saved artifact at '/tmp/tmpgz5tdqgy'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 35), dtype=tf.float32, name=None)
Captures:
  138012901222800: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  138012901225872: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  138013555880208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138013555880400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138013555875408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138013555881360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138013555882704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138013555881744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138013555882320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138013555883088: TensorSpec(shape=(), dtype=tf.resource, name=No